In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 중인 디바이스: {device}')

사용 중인 디바이스: cuda


In [18]:
BASE_DIR  = '/content/drive/MyDrive/dataset'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

train_df          = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df           = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
sample_submission = pd.read_csv(os.path.join(BASE_DIR, 'sample_submission.csv'))

print('=== train.csv ===')
print(train_df.head())
print(f'\n총 학습 데이터: {len(train_df)}개')
print(f'클래스 수: {train_df["label"].nunique()}개')
print('\n=== test.csv ===')
print(test_df.head())
print(f'\n총 테스트 데이터: {len(test_df)}개')

=== train.csv ===
  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6

총 학습 데이터: 723개
클래스 수: 10개

=== test.csv ===
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG

총 테스트 데이터: 199개


In [19]:
train_df['file_path'] = train_df['file_name'].apply(lambda x: os.path.join(TRAIN_DIR, x))
test_df['file_path']  = test_df['file_name'].apply(lambda x: os.path.join(TEST_DIR, x))

print('경로 생성 완료')
print(train_df[['file_name', 'file_path', 'label']].head())

경로 생성 완료
  file_name                                     file_path  label
0   001.PNG  /content/drive/MyDrive/dataset/train/001.PNG      9
1   002.PNG  /content/drive/MyDrive/dataset/train/002.PNG      4
2   003.PNG  /content/drive/MyDrive/dataset/train/003.PNG      1
3   004.PNG  /content/drive/MyDrive/dataset/train/004.PNG      1
4   005.PNG  /content/drive/MyDrive/dataset/train/005.PNG      6


In [20]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f'학습 데이터: {len(train_data)}개')
print(f'검증 데이터: {len(val_data)}개')

학습 데이터: 578개
검증 데이터: 145개


In [21]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print('전처리 설정 완료')

전처리 설정 완료


In [22]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df        = df
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'file_path']
        image    = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image

        label = self.df.loc[idx, 'label']
        return image, label

print('CustomDataset 정의 완료')

CustomDataset 정의 완료


In [23]:
BATCH_SIZE = 32

train_dataset = CustomDataset(train_data, transform=train_transform)
val_dataset   = CustomDataset(val_data,   transform=val_transform)
test_dataset  = CustomDataset(test_df,    transform=val_transform, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train_loader: {len(train_loader)}개 배치')
print(f'val_loader:   {len(val_loader)}개 배치')
print(f'test_loader:  {len(test_loader)}개 배치')

images, labels = next(iter(train_loader))
print(f'\n배치 이미지 shape: {images.shape}')
print(f'배치 라벨 shape:   {labels.shape}')

train_loader: 19개 배치
val_loader:   5개 배치
test_loader:  7개 배치

배치 이미지 shape: torch.Size([32, 3, 224, 224])
배치 라벨 shape:   torch.Size([32])


In [ ]:
# CNN 모델 정의
# =========================================================

class CNNModel(nn.Module):

    def __init__(self, num_classes):

        super(CNNModel, self).__init__()

        # 특징 추출 부분
        self.features = nn.Sequential(

            # Conv Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Conv Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Conv Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # 분류 부분
        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 28 * 28, 256),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
# Device 설정
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

In [ ]:
# 모델 생성
# =========================================================

# 클래스 개수 자동 계산
num_classes = train_df['label'].nunique()

model = CNNModel(num_classes).to(device)

print(model)

In [ ]:
# Loss Function / Optimizer 정의
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


In [ ]:
# 모델 학습 및 검증
# =========================================================

epochs = 10

for epoch in range(epochs):

    # -------------------------------
    # train mode
    # -------------------------------

    model.train()

    train_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        # gradient 초기화
        optimizer.zero_grad()

        # forward
        outputs = model(images)

        # loss 계산
        loss = criterion(outputs, labels)

        # backward
        loss.backward()

        # weight 업데이트
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f}")

    # -------------------------------
    # validation mode
    # -------------------------------

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # 가장 높은 확률의 클래스 선택
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total

    print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Test 데이터 예측
# =========================================================

model.eval()

predictions = []

with torch.no_grad():

    for images in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())


In [ ]:
# Submission 파일 생성
# =========================================================

sample_submission['label'] = predictions

sample_submission.to_csv(
    "submission_cnn.csv",
    index=False
)

print("\nsubmission_cnn.csv 저장 완료")